# SAHOO — Kaggle-compatible runner

This notebook adapts the SAHOO recursive self-improvement experiment to Kaggle. It keeps the original GDI/CPS metrics, RSI cycle logic, reporting, and post-hoc analysis, while adding dependency handling, Kaggle GPU settings, 4-bit NF4 model loading, a small smoke-test profile, and secret-based Hugging Face authentication.

**Start with `RUN_MODE="smoke"`.** This runs 2 samples per dataset and 1 improvement cycle. The original full setting covers 189 tasks and up to 20 cycles, which can exceed Kaggle's available GPU time. A smoke run checks the software path only; it is not a paper-scale replication.

In Kaggle, enable **GPU** and **Internet**. Add a Hugging Face token as a Kaggle Secret named `HF_TOKEN` only when required. The notebook does not contain or print credentials.

**Credential warning:** the upstream repository included a hard-coded Hugging Face token in its notebook and `config.json`. Revoke that token in Hugging Face and create a replacement before using it.

Upstream source: https://github.com/SubramanyamSahoo/SAHOO-Safeguarded-Alignment-for-High-Order-Optimization-Objectives-in-Recursive-Self-Improvement


In [ ]:
# CELL 1 — Runtime configuration (edit these values before running)
import os, sys, json, gc, math, random, logging, warnings, subprocess
from pathlib import Path
from datetime import datetime

# Kaggle working directory persists as notebook output; temporary cache stays local.
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_ROOT = WORK_DIR / "sahoo_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Start with a smoke test. Recommended first run: 2 samples x 1 cycle per domain.
RUN_MODE = "smoke"          # "smoke" or "full"
MODEL_NAME = "Qwen/Qwen3-8B"
USE_4BIT = True              # 4-bit NF4 quantization for smaller Kaggle GPUs.
MAX_NEW_TOKENS_SMOKE = 128
MAX_NEW_TOKENS_FULL = 512
SAMPLES_PER_DATASET_SMOKE = 2
SAMPLES_PER_DATASET_FULL = 63
MAX_CYCLES_SMOKE = 1
MAX_CYCLES_FULL = 20
BOOTSTRAP_SAMPLES = 300 if RUN_MODE == "smoke" else 2000
SEED = 42

# Optional: set HF_TOKEN in Kaggle Secrets with the exact name HF_TOKEN.
# Never paste a token into a notebook or commit it to GitHub.
try:
    from kaggle_secrets import UserSecretsClient
    _secret = UserSecretsClient().get_secret("HF_TOKEN")
    if _secret:
        os.environ["HF_TOKEN"] = _secret
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("HF_HOME", str(WORK_DIR / ".cache" / "huggingface"))
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

assert RUN_MODE in {"smoke", "full"}
SAMPLES_PER_DATASET = SAMPLES_PER_DATASET_SMOKE if RUN_MODE == "smoke" else SAMPLES_PER_DATASET_FULL
MAX_CYCLES = MAX_CYCLES_SMOKE if RUN_MODE == "smoke" else MAX_CYCLES_FULL
MAX_NEW_TOKENS = MAX_NEW_TOKENS_SMOKE if RUN_MODE == "smoke" else MAX_NEW_TOKENS_FULL

print({"run_mode": RUN_MODE, "model": MODEL_NAME, "4bit": USE_4BIT,
       "samples_per_dataset": SAMPLES_PER_DATASET, "max_cycles": MAX_CYCLES,
       "max_new_tokens": MAX_NEW_TOKENS, "output_root": str(OUTPUT_ROOT)})


In [ ]:
# CELL 2 — Install only missing runtime dependencies; do not reinstall PyTorch.
import importlib.util
required = {
    "transformers": "transformers>=4.45.0",
    "datasets": "datasets>=2.20.0",
    "accelerate": "accelerate>=0.34.0",
    "bitsandbytes": "bitsandbytes>=0.43.3",
    "sentencepiece": "sentencepiece>=0.2.0",
    "scipy": "scipy>=1.11.0",
    "sklearn": "scikit-learn>=1.3.0",
    "seaborn": "seaborn>=0.13.0",
}
missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Dependencies already available; skipping pip install.")

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.spatial.distance import cosine, jensenshannon
from scipy.special import rel_entr
from scipy.optimize import minimize_scalar
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance, MinCovDet
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig
from dataclasses import dataclass, field, asdict
from enum import Enum, auto
from abc import ABC, abstractmethod
from collections import defaultdict
from typing import Dict, List, Any, Optional, Tuple, Union, Callable
import hashlib, copy
warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
else:
    gpu_name, gpu_gb = "CPU", 0.0
print(f"Python={sys.version.split()[0]} | PyTorch={torch.__version__} | GPU={gpu_name} ({gpu_gb:.1f} GB)")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU in Settings > Accelerator before running the model cells.")


## Original SAHOO framework
The implementation is loaded from the linked public repository at runtime. Kaggle Internet must be enabled. The loader keeps the original classes and algorithms, but skips duplicate installs and hard-coded `__main__` blocks.


In [ ]:
# CELL 3 — Download and load the original SAHOO implementation safely
# Kaggle Settings > Internet must be enabled. This downloads the public source notebook.
import requests, re, ast

SOURCE_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/"
    "SubramanyamSahoo/SAHOO-Safeguarded-Alignment-for-High-Order-Optimization-Objectives-in-Recursive-Self-Improvement/"
    "main/Recursive%20Self%20Improvemment%20_ICLR%2026.ipynb"
)
resp = requests.get(SOURCE_NOTEBOOK_URL, timeout=90)
resp.raise_for_status()
source_nb = resp.json()
source_cells = {i: "".join(c.get("source", [])) for i, c in enumerate(source_nb.get("cells", []))}
expected_cells = {4, 9, 11, 12, 13, 15, 16, 17, 18, 24}
missing_cells = expected_cells.difference(source_cells)
if missing_cells:
    raise RuntimeError(f"Upstream notebook structure changed; missing cells: {sorted(missing_cells)}")

# Preserve the original order. The source notebook has empty cells and duplicate setup cells.
# Cell 4: imports and environment check (do not run its duplicate pip installs).
env_source = source_cells[4]
imports_start = env_source.find("# Core imports")
if imports_start < 0:
    raise RuntimeError("Could not find core imports in upstream environment cell 4.")
# Keep only imports from the upstream cell, excluding hardware checks and config initialization.
env_source = env_source[imports_start:]
imports_end = env_source.find('print("=" * 80)')
if imports_end >= 0:
    env_source = env_source[:imports_end]

# Cell 9 duplicates package installation and imports. Keep only its scientific/HF imports.
framework_source = source_cells[9]
framework_imports_start = framework_source.find("# Core imports")
if framework_imports_start < 0:
    raise RuntimeError("Could not find core imports in upstream framework cell 9.")
framework_source = framework_source[framework_imports_start:]
# Keep the framework header and imports only; config/dataclass definitions are in cell 11.
framework_end = framework_source.find('print("=" * 80)')
if framework_end >= 0:
    framework_source = framework_source[:framework_end]

# Cell 11 defines hardware/config objects. Remove the H100/CUDA hard stop; actual CUDA check is earlier.
config_source = source_cells[11]
config_source = re.sub(
    r'if not hardware\.cuda_available:\s*raise RuntimeError\([^\n]*\)\s*',
    '', config_source
)
config_source = re.sub(
    r'(?ms)^# Initialize configuration\s*\nconfig = ExperimentConfig\(\)\s*\nconfig\.save\(\)\s*',
    '', config_source
)

# Cell 12 uses ExperimentConfig and defines logger + datasets + ReasoningModel.
# Cell 13 defines drift, constraint and quality metrics.
# Cell 15 defines RSI engine/orchestrator.
# Cell 16/17 define charts/reports. Cell 18 defines run_experiment.
# Cell 24 defines post-hoc analysis (remove stale demo block at the bottom).
posthoc_source = source_cells[24]
if "# Quick test (run after experiment completes)" in posthoc_source:
    posthoc_source = posthoc_source.split("# Quick test (run after experiment completes)", 1)[0]

# The original main cell has a hard-coded HF token; strip the entire __main__ launcher.
main_source = source_cells[18]
main_source = re.sub(
    r'\n# Execute experiment\s*\nif __name__\s*==\s*["\']__main__["\']\s*:\s*.*$',
    '\n', main_source, flags=re.S
)
main_source = main_source.replace(
    "    config = ExperimentConfig()\n    config.save()",
    "    config = KAGGLE_EXPERIMENT_CONFIG\n    config.save()"
)

ordered = [
    ("upstream_imports_cell4", env_source),
    ("upstream_imports_cell9", framework_source),
    ("upstream_config_cell11", config_source),
    ("upstream_logger_dataset_model_cell12", source_cells[12]),
    ("upstream_metrics_cell13", source_cells[13]),
    ("upstream_engine_cell15", source_cells[15]),
    ("upstream_visualization_cell16", source_cells[16]),
    ("upstream_report_cell17", source_cells[17]),
    ("upstream_main_definition_cell18", main_source),
    ("upstream_posthoc_cell24", posthoc_source),
]

# Build execution plan so config exists before cell 12 creates the module-level logger.
execution_plan = []
for label, src in ordered:
    execution_plan.append((label, src))
    if label == "upstream_config_cell11":
        config_setup = "\n".join([
            "config = ExperimentConfig()",
            "config.model.name = MODEL_NAME",
            "config.model.max_new_tokens = MAX_NEW_TOKENS",
            'config.model.torch_dtype = "float16" if "T4" in gpu_name else ("bfloat16" if torch.cuda.is_bf16_supported() else "float16")',
            "config.dataset.samples_per_dataset = SAMPLES_PER_DATASET",
            "config.dataset.seed = SEED",
            "config.rsi.max_improvement_cycles = MAX_CYCLES",
            "config.metrics.bootstrap_samples = BOOTSTRAP_SAMPLES",
            'config.output_dir = str(OUTPUT_ROOT / f"{RUN_MODE}_{datetime.now():%Y%m%d_%H%M%S}")',
            "Path(config.output_dir).mkdir(parents=True, exist_ok=True)",
            'config.hf_token = os.environ.get("HF_TOKEN")',
            "KAGGLE_EXPERIMENT_CONFIG = config",
            "config.save()",
        ])
        execution_plan.append(("kaggle_config_setup", config_setup))

for label, src in execution_plan:
    compile(src, f"<{label}>", "exec")
for label, src in execution_plan:
    exec(compile(src, f"<{label}>", "exec"), globals(), globals())

print("Loaded original SAHOO framework definitions.")
print("Note: the upstream HF token-containing main block was intentionally not executed.")


## Run the experiment

Run cells from top to bottom. Keep `RUN_MODE="smoke"` until the test completes; then change it to `full` in Cell 1 and restart the session.


In [ ]:
# CELL 4 — Apply Kaggle settings, patch model loading, and run
import inspect

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle notebook settings before loading the model.")

# The source loader has already created and applied the Kaggle configuration before logger initialization.
config = KAGGLE_EXPERIMENT_CONFIG
config.hf_token = os.environ.get("HF_TOKEN")
if not config.hf_token:
    print("HF_TOKEN not set. Continuing anonymously; set Kaggle Secret HF_TOKEN if Hugging Face authentication is required.")

# Enable 4-bit quantization for the 8B model on smaller Kaggle GPUs.
# bitsandbytes must be available from Cell 2.
if not hasattr(ReasoningModel, "_initialize"):
    raise RuntimeError("The upstream ReasoningModel._initialize method was not found; model loader patch cannot be applied.")
_original_initialize = ReasoningModel._initialize

def _kaggle_initialize(self):
    dtype = torch.float16 if self.config.model.torch_dtype == "float16" else torch.bfloat16
    self.tokenizer = AutoTokenizer.from_pretrained(
        self.config.model.name,
        trust_remote_code=self.config.model.trust_remote_code,
        token=self.config.hf_token,
    )
    if self.tokenizer.pad_token is None:
        self.tokenizer.pad_token = self.tokenizer.eos_token
    self.tokenizer.padding_side = "left"

    model_kwargs = {
        "device_map": "auto",
        "trust_remote_code": self.config.model.trust_remote_code,
        "token": self.config.hf_token,
    }
    if USE_4BIT:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs["torch_dtype"] = dtype

    self.model = AutoModelForCausalLM.from_pretrained(
        self.config.model.name,
        **model_kwargs,
    )
    self.model.eval()
    self._embedding_dim = self.model.config.hidden_size
    self._vocab_size = self.model.config.vocab_size
    total_params = sum(p.numel() for p in self.model.parameters())
    self.logger.info(f"Model loaded: {total_params / 1e9:.2f}B parameters; 4-bit={USE_4BIT}")
    self.logger.info(f"Embedding dim: {self._embedding_dim}; vocab size: {self._vocab_size}")

ReasoningModel._initialize = _kaggle_initialize

config.save()
print("Run profile:", {
    "model": config.model.name,
    "dtype": config.model.torch_dtype,
    "4bit": USE_4BIT,
    "samples_per_dataset": config.dataset.samples_per_dataset,
    "cycles": config.rsi.max_improvement_cycles,
    "max_new_tokens": config.model.max_new_tokens,
    "output_dir": config.output_dir,
})

experiment_output = run_experiment()


In [ ]:
# CELL 5 — Post-experiment analysis (no expensive ablation re-runs)
OUTPUT_DIR = experiment_output["output_dir"]
posthoc_results = run_post_experiment_analysis(
    OUTPUT_DIR,
    run_ablations=False,
    run_theoretical=True,
)


In [ ]:
# CELL — List generated artefacts
from IPython.display import display
out_dir = Path(OUTPUT_DIR)
for p in sorted(out_dir.iterdir()):
    print(f"{p.name:40s} {p.stat().st_size/1024:10.1f} KB")
